# Cascade decays

:::{autolink-concat}
:::

The {mod}`ampform.amplitude.cascade` module formulates helicity amplitude models for decays with **any number of final-state particles** and arbitrary decay depth, following the conventions of {cite}`Habermann:2024sxs`. Amplitudes for decay chains with different topologies are matched by rotating each final-state spin state with a **relative Wigner rotation** with respect to a reference topology,

$$
\mathcal{A}_{\lambda_0,\{\lambda\}} =
  \sum_{\mathfrak{c}} \sum_{\{\lambda'\}}
  \mathcal{A}^{\mathfrak{c}}_{\lambda_0,\{\lambda'\}}
  \prod_i D^{j_i*}_{\lambda'_i,\lambda_i}\!\left(
    \alpha_i^\mathfrak{c}, \beta_i^\mathfrak{c}, \gamma_i^\mathfrak{c}
  \right),
$$

where each chain amplitude is a product of one Wigner-$D$ function and one helicity coupling per two-body decay node,

$$
\mathcal{A}^{\mathfrak{c}}_{\lambda_0,\{\lambda\}} =
  \sum_{\lambda_\text{internal}} \prod_\kappa
  D^{j_\kappa*}_{\lambda_\kappa,\lambda_{\kappa_1}-\lambda_{\kappa_2}}\!\left(
    \phi_\kappa, \theta_\kappa, 0
  \right)
  \mathcal{H}^{\kappa}_{\lambda_{\kappa_1},\lambda_{\kappa_2}}
  \, (-1)^{j_{\kappa_2} - \lambda_{\kappa_2}}
  \, X_\kappa\!\left(m_\kappa\right).
$$

Unlike in {doc}`Dalitz-plot decomposition </dpd>`, the angles in these Wigner-$D$ functions **cannot be expressed in terms of Mandelstam variables**. They remain free symbols in the model and have to be computed numerically from four-momenta with the [`decayangle`](https://pypi.org/project/decayangle) package, which implements the same conventions. Each symbol in {attr}`.CascadeAmplitudeModel.kinematic_variables` carries a descriptor ({class}`.HelicityAngle`, {class}`.WignerRotation`, or {class}`~ampform.amplitude.cascade.InvariantMass`) that tells you exactly which `decayangle` function to call.

:::{note}
AmpForm itself does **not** depend on [`decayangle`](https://pypi.org/project/decayangle), nor on [`phasespace`](https://pypi.org/project/phasespace), which is used below to generate a phase-space sample. Both are dependencies of the `notebooks` dependency group only.
:::

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from IPython.display import Latex, Markdown

from ampform.amplitude.cascade import (
    CascadeAmplitudeBuilder,
    HelicityAngle,
    InvariantMass,
    OrientationAngle,
    WignerRotation,
    create_invariant_mass_symbol,
    to_topology,
)
from ampform.decay import (
    Decay,
    DecayChain,
    IsobarNode,
    Particle,
    State,
    get_final_state_ids,
)
from ampform.dynamics.builder import DefinedExpression
from ampform.io import aslatex, improve_latex_rendering, mute_ampform_warnings
from ampform.sympy import cached

improve_latex_rendering()
warnings.simplefilter("ignore")

if STATIC_PAGE := "EXECUTE_NB" in os.environ:
    mute_ampform_warnings()

## Decay definition

As an example, we take the four-body decay $\Lambda_c^+ \to p K^- \pi^+ \pi^0$ with three decay chains that cover two different topology types: a 'double-pair' topology, $\Lambda_c^+ \to \Delta(1232)^{++} K^*(892)^-$, and a deeper cascade through $\Sigma(1670)^0 \to \Lambda(1520)\pi^0$. Since [QRules](https://qrules.rtfd.io) formulates transitions for three-body decays, we construct the {class}`.DecayChain` objects by hand:

In [ ]:
Λc = State(
    "Lambda_c+", R"\Lambda_c^+", spin=0.5, parity=+1, mass=2.28646, width=0.0, index=0
)
p = State("p", "p", spin=0.5, parity=+1, mass=0.93827, width=0.0, index=1)
K = State("K-", "K^-", spin=0, parity=-1, mass=0.49368, width=0.0, index=2)
π = State("pi+", R"\pi^+", spin=0, parity=-1, mass=0.13957, width=0.0, index=3)
π0 = State("pi0", R"\pi^0", spin=0, parity=-1, mass=0.13498, width=0.0, index=4)

Δ = Particle(
    "Delta(1232)++", R"\Delta(1232)^{++}", spin=1.5, parity=+1, mass=1.232, width=0.117
)
Kstar = Particle("K*(892)-", R"K^*(892)^-", spin=1, parity=-1, mass=0.892, width=0.0514)
Λ1520 = Particle(
    "Lambda(1520)", R"\Lambda(1520)", spin=1.5, parity=-1, mass=1.51945, width=0.0157
)
Σ1670 = Particle(
    "Sigma(1670)0", R"\Sigma(1670)^0", spin=1.5, parity=-1, mass=1.670, width=0.06
)

DECAY = Decay(
    states={s.index: s for s in [Λc, p, K, π, π0]},
    chains=[
        DecayChain(
            decay=IsobarNode(
                parent=Λc,
                child1=IsobarNode(Δ, p, π),
                child2=IsobarNode(Kstar, K, π0),
            )
        ),
        DecayChain(
            decay=IsobarNode(
                parent=Λc,
                child1=IsobarNode(Σ1670, IsobarNode(Λ1520, p, K), π0),
                child2=π,
            )
        ),
    ],
)
Latex(
    aslatex({
        sp.Symbol(Rf"\text{{{to_topology(chain)}}}"): chain.decay
        for chain in DECAY.chains
    })
)

The keys in the table above are the **topology descriptors** of each chain (see {func}`.to_topology`). They can be passed directly to `decayangle`'s `Topology` class.

## Model formulation

Resonance lineshapes are registered per resonance name. A dynamics builder receives the {class}`.IsobarNode` of the decaying resonance and formulates its lineshape in terms of an invariant mass symbol, which remains a free variable that is later computed from the four-momenta:

In [ ]:
def formulate_breit_wigner(node: IsobarNode) -> DefinedExpression:
    resonance = node.parent
    m = create_invariant_mass_symbol(get_final_state_ids(node))
    m0 = sp.Symbol(Rf"m_{{{resonance.latex}}}", nonnegative=True)
    Γ0 = sp.Symbol(Rf"\Gamma_{{{resonance.latex}}}", nonnegative=True)
    return DefinedExpression(
        expression=1 / (m0**2 - m**2 - sp.I * m0 * Γ0),
        parameters={m0: resonance.mass, Γ0: resonance.width},
    )


model_builder = CascadeAmplitudeBuilder(DECAY)
for chain in DECAY.chains:
    for resonance in chain.resonances:
        model_builder.dynamics_choices.register_builder(
            resonance.name, formulate_breit_wigner
        )
model = model_builder.formulate(reference_topology=((1, 3), (2, 4)))
model.intensity

Each topology contributes one amplitude $A^{\mathfrak{c}}$ per helicity combination, with one conjugated Wigner-$D$ function, one helicity coupling $\mathcal{H}$, and a particle-2 phase per isobar node:

In [ ]:
half = sp.Rational(1, 2)
example_amplitudes = {
    symbol: expression
    for symbol, expression in model.amplitudes.items()
    if symbol.indices == (half, half, 0, 0, 0)
}
Latex(aslatex(example_amplitudes))

## Computing the angles with decayangle

None of the kinematic symbols of the model have a symbolic definition. Instead, {attr}`.CascadeAmplitudeModel.kinematic_variables` maps each symbol to a descriptor of how to compute it numerically:

In [ ]:
rows = [
    f"| ${sp.latex(symbol)}$ | `{descriptor!r}` |"
    for symbol, descriptor in model.kinematic_variables.items()
]
Markdown("| Symbol | Descriptor |\n|---|---|\n" + "\n".join(rows))

The descriptors translate mechanically into calls to `decayangle`, given a mapping of final-state IDs to four-momenta arrays of the form $(p_x, p_y, p_z, E)$ in the rest frame of the $\Lambda_c^+$:

In [ ]:
from decayangle.decay_topology import Topology


def compute_kinematic_variables(
    model, momenta: dict[int, np.ndarray]
) -> dict[sp.Symbol, np.ndarray]:
    """Compute all kinematic variables of a cascade model with decayangle."""
    data = {}
    helicity_cache = {}
    wigner_cache = {}
    for symbol, variable in model.kinematic_variables.items():
        if isinstance(variable, HelicityAngle):
            angles = helicity_cache.get(variable.topology)
            if angles is None:
                topology = Topology(0, decay_topology=variable.topology)
                angles = topology.helicity_angles(momenta)
                helicity_cache[variable.topology] = angles
            data[symbol] = getattr(angles[variable.node], variable.angle)
        elif isinstance(variable, WignerRotation):
            key = (variable.reference_topology, variable.topology)
            angles = wigner_cache.get(key)
            if angles is None:
                reference = Topology(0, decay_topology=variable.reference_topology)
                topology = Topology(0, decay_topology=variable.topology)
                angles = reference.relative_wigner_angles(topology, momenta)
                wigner_cache[key] = angles
            data[symbol] = getattr(angles[variable.state_id], variable.angle)
        elif isinstance(variable, InvariantMass):
            p = np.sum([momenta[i] for i in variable.state_ids], axis=0)
            data[symbol] = np.sqrt(p[..., 3] ** 2 - np.sum(p[..., :3] ** 2, axis=-1))
    return data

We generate a phase-space sample for the four-body decay with [`phasespace`](https://pypi.org/project/phasespace), which conveniently returns the four-momenta in the required $(p_x, p_y, p_z, E)$ order and in the rest frame of the decaying particle:

In [ ]:
import phasespace

FINAL_STATE_IDS = sorted(DECAY.final_state)
phsp_weights, phsp_sample = phasespace.nbody_decay(
    DECAY.initial_state.mass,
    [DECAY.final_state[i].mass for i in FINAL_STATE_IDS],
).generate(n_events=100_000, seed=0)
momenta = {i: phsp_sample[f"p_{i - 1}"].numpy() for i in FINAL_STATE_IDS}
phsp_weights = phsp_weights.numpy()
kinematic_data = compute_kinematic_variables(model, momenta)

## Intensity distribution

We substitute all fixed parameters (resonance masses and widths) into the expression, but keep the helicity couplings as **arguments** of the lambdified function, so that they remain tunable:

In [ ]:
def create_intensity_func(model):
    expression = cached.unfold(model)
    fixed_parameters = {
        symbol: value
        for symbol, value in model.parameter_defaults.items()
        if not str(symbol).startswith(R"\mathcal{H}")
    }
    expression = cached.xreplace(expression, fixed_parameters)
    couplings = sorted(
        (
            symbol
            for symbol in model.parameter_defaults
            if str(symbol).startswith(R"\mathcal{H}")
        ),
        key=str,
    )
    kinematic_symbols = sorted(model.kinematic_variables, key=str)
    func = sp.lambdify([*kinematic_symbols, *couplings], expression, "numpy", cse=True)
    coupling_values = [complex(model.parameter_defaults[s]) for s in couplings]

    def evaluate(data: dict) -> np.ndarray:
        arguments = [data[symbol] for symbol in kinematic_symbols]
        return np.real(func(*arguments, *coupling_values))

    return evaluate


intensity_func = create_intensity_func(model)
intensities = intensity_func(kinematic_data)
assert not np.any(np.isnan(intensities))

In [ ]:
%config InlineBackend.figure_formats = ['svg']

In [ ]:
def invariant_mass(*state_ids: int) -> np.ndarray:
    p = sum(momenta[i] for i in state_ids)
    return np.sqrt(p[..., 3] ** 2 - np.sum(p[..., :3] ** 2, axis=-1))


subsystems = {
    (1, 3): R"$m\left(p\pi^+\right)$ [GeV]",
    (2, 4): R"$m\left(K^-\pi^0\right)$ [GeV]",
    (1, 2): R"$m\left(pK^-\right)$ [GeV]",
    (1, 2, 4): R"$m\left(pK^-\pi^0\right)$ [GeV]",
}
plt.rc("font", size=12)
fig, axes = plt.subplots(figsize=(11, 7), ncols=2, nrows=2, layout="constrained")
for ax, (ids, label) in zip(axes.flatten(), subsystems.items(), strict=True):
    mass = invariant_mass(*ids)
    bins = 100
    ax.hist(
        mass,
        bins=bins,
        weights=phsp_weights,
        density=True,
        alpha=0.5,
        label="phase space",
    )
    ax.hist(
        mass,
        bins=bins,
        weights=phsp_weights * intensities,
        density=True,
        histtype="step",
        lw=2,
        label="intensity",
    )
    ax.set_xlabel(label)
    ax.set_ylim(bottom=0)
axes[0, 0].legend()
plt.show()

The $\Delta(1232)^{++}$, $K^*(892)^-$, $\Lambda(1520)$, and $\Sigma(1670)^0$ resonances are clearly visible in their respective subsystems.

## Reference topology independence

As a cross-check of the alignment conventions, the intensity must be **independent of the choice of reference topology**: choosing the cascade topology as reference instead only reshuffles the Wigner rotations, but describes the same physics. This only holds if the final-state spin states of all topologies are matched correctly, so this check validates the entire chain of conventions between AmpForm and `decayangle`:

In [ ]:
alternative_model = model_builder.formulate(reference_topology=(((1, 2), 4), 3))
alternative_func = create_intensity_func(alternative_model)
subset = {i: momentum[:10_000] for i, momentum in momenta.items()}
alternative_intensities = alternative_func(
    compute_kinematic_variables(alternative_model, subset)
)
np.testing.assert_allclose(
    alternative_intensities,
    intensities[:10_000],
    rtol=1e-10,
)

:::{seealso}
For three-body decays, the alignment angles *can* be expressed in Mandelstam variables, which makes the fully symbolic {doc}`Dalitz-plot decomposition </dpd>` a better choice.
:::

## Spin-density matrix

Just like for {doc}`three-body decays </dpd/polarization>`, the intensity of a **polarized** initial state can be written as a matrix expression,

$$
I = \operatorname{tr}\left(\rho\,\overline{D}\,O\,\overline{D}^\dagger\right),
\qquad
O_{\nu,\nu'} = \sum_{\{\lambda\}}
    \mathcal{A}_{\nu,\{\lambda\}}\,\mathcal{A}^*_{\nu',\{\lambda\}}\,,
$$

with $\rho$ the spin-density matrix of the $\Lambda_c^+$ in the production frame and $\overline{D}$ a matrix of conjugated Wigner-$D$ functions over the Euler angles $(\phi, \theta, \chi)$ that orient the decay relative to the production frame. For an $n$-body decay, these orientation angles are defined by a **reference subsystem**, which fixes the plane of the first three-body decay of the reference topology: the aligned frame has the reference-subsystem momentum along the $z$-axis and the momentum of its first daughter in the $xz$-half-plane (see {class}`.OrientationAngle`). By default, the reference subsystem is the first daughter of the root node, here the $\left(p\pi^+\right)$ system:

In [ ]:
polarized_model = model_builder.formulate(
    reference_topology=((1, 3), (2, 4)),
    polarized=True,
)
polarized_model.intensity

In [ ]:
Latex(aslatex(polarized_model.matrices))

The elements of the decay matrix $O$ are formulated in terms of the same topology amplitudes as before. The variables of a polarized model split into two groups: the orientation angles $(\phi, \theta, \chi)$ are computed from the momenta **in the production frame**, while all remaining kinematic variables have to be computed over **aligned momenta**, from which the overall orientation has been rotated away:

In [ ]:
def compute_orientation_angles(
    model, momenta: dict[int, np.ndarray]
) -> dict[str, np.ndarray]:
    """Compute the Euler angles that orient the decay, following `OrientationAngle`."""
    descriptor = next(
        descriptor
        for descriptor in model.kinematic_variables.values()
        if isinstance(descriptor, OrientationAngle)
    )
    p1 = np.sum([momenta[i] for i in descriptor.subsystem_ids], axis=0)
    p2 = np.sum([momenta[i] for i in descriptor.daughter_ids], axis=0)
    φ = np.arctan2(p1[..., 1], p1[..., 0])
    θ = np.arccos(p1[..., 2] / np.linalg.norm(p1[..., :3], axis=-1))
    p2_rotated = rotate_y(rotate_z(p2, -φ), -θ)
    χ = np.arctan2(p2_rotated[..., 1], p2_rotated[..., 0])
    return {"phi": φ, "theta": θ, "chi": χ}


def align_momenta(
    momenta: dict[int, np.ndarray], orientation: dict[str, np.ndarray]
) -> dict[int, np.ndarray]:
    """Rotate the momenta into the aligned frame of the reference subsystem."""
    φ, θ, χ = (orientation[angle] for angle in ("phi", "theta", "chi"))
    return {
        i: rotate_z(rotate_y(rotate_z(momentum, -φ), -θ), -χ)
        for i, momentum in momenta.items()
    }


def rotate_z(p: np.ndarray, angle) -> np.ndarray:
    p = np.array(p, dtype=float)
    x, y = p[..., 0].copy(), p[..., 1].copy()
    p[..., 0] = np.cos(angle) * x - np.sin(angle) * y
    p[..., 1] = np.sin(angle) * x + np.cos(angle) * y
    return p


def rotate_y(p: np.ndarray, angle) -> np.ndarray:
    p = np.array(p, dtype=float)
    x, z = p[..., 0].copy(), p[..., 2].copy()
    p[..., 0] = np.cos(angle) * x + np.sin(angle) * z
    p[..., 2] = -np.sin(angle) * x + np.cos(angle) * z
    return p


orientation = compute_orientation_angles(polarized_model, momenta)
aligned_momenta = align_momenta(momenta, orientation)
polarized_data = compute_kinematic_variables(polarized_model, aligned_momenta)
polarized_data.update({
    symbol: orientation[descriptor.angle]
    for symbol, descriptor in polarized_model.kinematic_variables.items()
    if isinstance(descriptor, OrientationAngle)
})

The elements of $\rho$ enter as parameters, with the unpolarized case $\rho = \mathbb{1}/2$ as default. We keep them as arguments of the lambdified function, next to the helicity couplings:

In [ ]:
def create_polarized_func(model):
    expression = cached.unfold(model)
    fixed_parameters = {
        symbol: value
        for symbol, value in model.parameter_defaults.items()
        if not str(symbol).startswith((R"\mathcal{H}", R"\rho"))
    }
    expression = cached.xreplace(expression, fixed_parameters)
    arguments = sorted(
        (
            symbol
            for symbol in model.parameter_defaults
            if str(symbol).startswith((R"\mathcal{H}", R"\rho"))
        ),
        key=str,
    )
    kinematic_symbols = sorted(model.kinematic_variables, key=str)
    func = sp.lambdify([*kinematic_symbols, *arguments], expression, "numpy", cse=True)

    def evaluate(data: dict, parameters: dict) -> np.ndarray:
        values = [
            complex(parameters.get(symbol, model.parameter_defaults[symbol]))
            for symbol in arguments
        ]
        return np.real(func(*(data[s] for s in kinematic_symbols), *values))

    return evaluate


polarized_func = create_polarized_func(polarized_model)

Two consistency checks. First, with the unpolarized $\rho = \mathbb{1}/2$, the trace collapses to half the unpolarized intensity computed earlier, independent of the orientation angles. Second, the factorization into $\overline{D}(\phi, \theta, \chi)$ and aligned decay matrix $O$ is exact: evaluating the same model over the **unaligned** momenta with $(\phi,\theta,\chi) = 0$ gives identical values, for any $\rho$:

In [ ]:
ρ = sp.IndexedBase(R"\rho")
half = sp.Rational(1, 2)
unpolarized_intensities = polarized_func(polarized_data, parameters={})
np.testing.assert_allclose(2 * unpolarized_intensities, intensities, rtol=1e-10)

polarization = {
    ρ[-half, -half]: 0.15,
    ρ[-half, +half]: 0.2 + 0.1j,
    ρ[+half, +half]: 0.85,
}
lab_data = compute_kinematic_variables(polarized_model, momenta)
zeros = np.zeros_like(orientation["phi"])
lab_data.update({
    symbol: zeros
    for symbol, descriptor in polarized_model.kinematic_variables.items()
    if isinstance(descriptor, OrientationAngle)
})
np.testing.assert_allclose(
    polarized_func(polarized_data, parameters=polarization),
    polarized_func(lab_data, parameters=polarization),
    rtol=1e-10,
)

Finally, the effect of polarization on the orientation of the decay. A longitudinally polarized $\Lambda_c^+$ with polarization $P_z$ has $\rho = \frac{1}{2}\operatorname{diag}(1-P_z,\, 1+P_z)$. Since the weak decay violates parity, the polarization shows up as an asymmetry in $\cos\theta$, the polar angle of the reference subsystem, which vanishes in the unpolarized case. The analyzing power depends on the helicity couplings, so we choose non-trivial example values for them (with all couplings equal, the asymmetry cancels):

In [ ]:
rng = np.random.default_rng(seed=1)
example_couplings = {
    symbol: complex(*rng.normal(size=2))
    for symbol in polarized_model.parameter_defaults
    if str(symbol).startswith(R"\mathcal{H}")
}
P_z = 0.9
scenarios = {
    R"$\rho = \mathbb{1}/2$ (unpolarized)": {},
    Rf"$P_z = {P_z}$": {
        ρ[-half, -half]: (1 - P_z) / 2,
        ρ[+half, +half]: (1 + P_z) / 2,
    },
}
fig, axes = plt.subplots(figsize=(11, 4.5), ncols=2, sharey=True, layout="constrained")
for title, parameters in scenarios.items():
    weights = phsp_weights * polarized_func(
        polarized_data, parameters={**example_couplings, **parameters}
    )
    axes[0].hist(
        np.cos(orientation["theta"]),
        bins=50,
        weights=weights,
        density=True,
        histtype="step",
        lw=2,
        label=title,
    )
    axes[1].hist(
        orientation["chi"],
        bins=50,
        weights=weights,
        density=True,
        histtype="step",
        lw=2,
        label=title,
    )
axes[0].set_xlabel(R"$\cos\theta$")
axes[1].set_xlabel(R"$\chi$")
axes[1].set_xticks([-np.pi, 0, +np.pi])
axes[1].set_xticklabels([R"$-\pi$", "0", R"$+\pi$"])
for ax in axes:
    ax.set_ylim(bottom=0)
axes[0].legend()
plt.show()